# Merit-Function History with HistoryObserver


`HistoryObserver` records the merit value at each accepted iteration. After the run, the per-iteration records are available as `result.history` (a list of dicts with keys `iteration` and `value`) and also on `observer.records` directly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optiland import optic
from optiland.optimization import minimize, OptimizationProblem
from optiland.optimization.observers.history import HistoryObserver

lens = optic.Optic()
lens.surfaces.add(index=0, thickness=np.inf)
lens.surfaces.add(index=1, thickness=6, radius=25, material="N-BK7", is_stop=True)
lens.surfaces.add(index=2, thickness=3, radius=-25, material="N-SF11")
lens.surfaces.add(index=3, thickness=45, radius=-100)
lens.surfaces.add(index=4)
lens.set_aperture(aperture_type="EPD", value=10)
lens.fields.set_type("angle")
lens.fields.add(y=0.0)
lens.fields.add(y=0.7)
lens.wavelengths.add(value=0.4861)
lens.wavelengths.add(value=0.5876, is_primary=True)
lens.wavelengths.add(value=0.6563)
lens.update_paraxial()


In [ ]:
problem = OptimizationProblem()
for field in lens.fields.get_field_coords():
    input_data = {"optic": lens, "surface_number": -1, "Hx": field[0], "Hy": field[1],
                  "num_rays": 5, "wavelength": 0.5876, "distribution": "hexapolar"}
    problem.add_operand("rms_spot_size", target=0, weight=1, input_data=input_data)
problem.add_variable(lens, "radius", surface_number=1)
problem.add_variable(lens, "radius", surface_number=2)
problem.add_variable(lens, "radius", surface_number=3)
problem.add_variable(lens, "thickness", surface_number=3)


In [ ]:
observer = HistoryObserver()
result = minimize(problem, "dls", observers=[observer], disp=False)
print(result)


In [ ]:
# result.history is a list of dicts: [{"iteration": 0, "value": ...}, ...]
history = result.history
iters = [rec["iteration"] for rec in history]
values = [rec["value"] for rec in history]

plt.figure(figsize=(8, 4))
plt.semilogy(iters, values)
plt.xlabel("Iteration")
plt.ylabel("Merit (log scale)")
plt.title("Merit-function convergence")
plt.grid(True)
plt.tight_layout()
plt.show()


The `records` attribute on the observer itself holds the same data as `result.history`: `observer.records` is a list of dicts with keys `iteration` and `value` (and optionally `step_norm`).
